# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [2]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [3]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the F1 World Championship in 2010 driving for Red Bull Racing.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [4]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [5]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

The 2006 F1 World Championship was won by Fernando Alonso driving for the Renault team.


In [6]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

The 2019 F1 championship was won by Lewis Hamilton driving for Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [7]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton. 
Team: Mercedes. 
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [8]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [9]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


Few Shots for classification.


In [10]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Neutral


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

### Exercise 1 Title: Credit Card Basics — Zero‑Shot Prompt


In [11]:
context_user = [
    {
        'role': 'system',
        'content': """
You are an expert in personal finance. You speak with the calm, serious tone of an adult narrator, similar to Morgan Freeman. 
Your explanations are brief, direct, and grounded. No humor, no embellishment.

The user will ask about how to use a credit card as if it were a debit card and how to pay it in full on the statement date.
Respond with clarity and simplicity.
"""
    }
]

print(return_OAIResponse(
    "Explain how to use a credit card like a debit card and pay it fully on the statement date.",
    context_user
))

To use a credit card like a debit card, only spend what you can afford to pay off in full each month. Make purchases within your budget and avoid carrying a balance. On the statement date, pay the full balance shown on your credit card statement by the due date. Set up automatic payments or reminders to ensure you never miss a payment. This will help you avoid interest charges and maintain a healthy credit score.


### Exercise 2 World Cup History — One‑Shot Prompt

In [12]:
context_user = [
    {
        'role': 'system',
        'content': """
You are an expert in the history of the FIFA World Cup. 
You speak with the calm, serious tone of an adult narrator, similar to Morgan Freeman.
Your answers are concise, factual, and delivered with quiet authority.

Below is an example of how you respond:

USER:
Give me an example of how you speak.

ASSISTANT:
I speak with calm precision. I say only what matters, as if time has taught me the value of clarity.

Now follow the same tone and structure when answering the user's next question.
"""
    }
]

print(return_OAIResponse(
    "Who were the first four World Cup champions?",
    context_user
))

The first four FIFA World Cup champions were Uruguay in 1930, Italy in 1934, Italy again in 1938, and then Uruguay once more in 1950.


### Exercise 3 Buyer Persona Essentials — Few‑Shot Prompt

In [13]:
context_user = [
    {
        'role': 'system',
        'content': """
You are an expert in marketing and buyer persona creation. 
You speak with the calm, serious tone of an adult narrator, similar to Morgan Freeman.
Your explanations are brief, grounded, and intentional.

Here are examples of how you respond:

USER:
Give me an example of your tone.

ASSISTANT:
I speak slowly and with purpose. I offer only what is essential, letting each idea stand on its own.

USER:
Another example.

ASSISTANT:
Every explanation carries weight. I avoid noise and focus on what truly matters.

USER:
One more.

ASSISTANT:
Marketing begins with understanding. And understanding requires calm observation and precise words.

Follow this tone and structure when answering the user's next question.
"""
    }
]

print(return_OAIResponse(
    "Explain briefly what characteristics a buyer persona should have.",
    context_user
))

A buyer persona should be specific, detailed, and based on real data. It should include demographics, behaviors, goals, challenges, and motivations of your target audience.Creating empathy is key-- understand their needs deeply to tailor your marketing effectively.


## Summary of What We Did

In this exercise, we compared how a language model behaves under Zero‑Shot, One‑Shot, and Few‑Shot prompting. We used the same task in all three cases—explaining the characteristics of a buyer persona—and kept a consistent tone: calm, serious, adult, Morgan‑Freeman‑style narration.
Zero‑Shot
The model received only instructions, with no examples.
It followed the tone reasonably well, but the structure and style were less consistent. This shows that with Zero‑Shot prompting, the model relies mainly on its general training and may vary in how it responds.
One‑Shot
We added a single example of the desired tone.
The model immediately adapted, producing a more stable and coherent response. This demonstrates how even one example strongly influences the model’s style and structure.
Few‑Shot
We provided three examples with the same tone and style.
The model became highly consistent, closely imitating the rhythm, tone, and precision of the examples. This shows how Few‑Shot prompting allows the model to “learn” patterns during inference and reproduce them reliably.

Conclusion
The comparison clearly shows that:
- Zero‑Shot → The model follows instructions but may vary.
- One‑Shot → The model imitates the example and becomes more consistent.
- Few‑Shot → The model strongly aligns with the demonstrated style and structure.
This exercise helps students understand how examples shape model behavior and why shot‑based prompting is a powerful technique for controlling tone, format, and consistency.
